# Primer contacto con scikit-learn: una API para trabajar con datos

**Objetivo:** conocer cómo se organiza y se usa scikit-learn antes de profundizar en los algoritmos. En unos 15–20 minutos veremos varias cosas que permite hacer **sin entrenar un clasificador**: cargar datos de ejemplo, completar valores ausentes, escalar números, convertir texto en números y comparar elementos.

**Requisito:** `scikit-learn` instalado. 

Si falta, ejecuta `%pip install scikit-learn` en una celda y reinicia el kernel. No hacen falta archivos ni conexión a Internet para los ejemplos.


## 1. Un mapa rápido de la biblioteca

| Necesidad | Módulo | Ejemplo |
|---|---|---|
| Probar con datos incluidos | `sklearn.datasets` | `load_iris` |
| Preparar números y categorías | `sklearn.preprocessing` | `StandardScaler` |
| Completar valores ausentes | `sklearn.impute` | `SimpleImputer` |
| Convertir texto en variables | `sklearn.feature_extraction.text` | `CountVectorizer` |
| Comparar vectores | `sklearn.metrics.pairwise` | `cosine_similarity` |

Normalmente importamos **la herramienta concreta** desde su módulo. Muchas herramientas comparten la misma forma de uso: crear un objeto, configurarlo y llamar a sus métodos.


In [1]:
import sklearn
from sklearn.datasets import load_iris
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Versión de scikit-learn:', sklearn.__version__)

Versión de scikit-learn: 1.9.0


## 2. Datos incluidos: mirar antes de usar

`load_iris()` entrega un objeto con datos y metadatos. `X` suele representar una tabla de **filas = observaciones** y **columnas = características**. `y` contiene las etiquetas cuando existen; aquí solo las inspeccionaremos.


In [2]:
iris = load_iris()
X = iris.data
y = iris.target

print('Forma de X:', X.shape)
print('Forma de y:', y.shape)
print('Columnas:', iris.feature_names)
print('Primeras 3 filas:', X[:3])
print('Nombres de las clases:', iris.target_names)

Forma de X: (150, 4)
Forma de y: (150,)
Columnas: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Primeras 3 filas: [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]]
Nombres de las clases: ['setosa' 'versicolor' 'virginica']


## 3. La pauta de la API: crear → `fit` → `transform`

Un **transformador** aprende en `fit` lo necesario para transformar datos. Por ejemplo, `SimpleImputer` aprende la media de cada columna; no está aprendiendo a predecir una clase. `transform` aplica después esa regla. `fit_transform` hace ambas cosas de una vez sobre los mismos datos.

Los atributos calculados al hacer `fit` suelen acabar en guion bajo: `statistics_`, `mean_`, etc. Los parámetros que elegimos al crear el objeto no lo llevan.


In [ ]:
datos = [[10.0, 1.0], [20.0, float('nan')], [30.0, 3.0]]

imputador = SimpleImputer(strategy='mean')  # parámetro elegido por nosotros
imputador.fit(datos)                          # calcula las medias por columna

print('Medias aprendidas:', imputador.statistics_)
print('Datos completados:', imputador.transform(datos))
print('Nueva fila completada:', imputador.transform([[40.0, float('nan')]]))

El mismo patrón sirve para escalar números. Conviene **ajustar una vez** el transformador y reutilizarlo para nuevas filas: así ambas pasan por la misma regla. En un proyecto predictivo, el ajuste se haría solo con los datos de entrenamiento para evitar fugas de información.


In [ ]:
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)

print('Media de cada columna original:', escalador.mean_.round(2))
print('Primera fila original:', X[0])
print('Primera fila escalada:', X_escalado[0].round(2))
print('Una fila nueva, con la misma regla:', escalador.transform([[5.1, 3.5, 1.4, 0.2]]).round(2))

## 4. Los parámetros también tienen una API común

`get_params()` deja ver la configuración y `set_params()` permite cambiarla. Después de cambiar un parámetro que afecta al cálculo, hay que volver a hacer `fit`.


In [ ]:
print('Estrategia inicial:', imputador.get_params()['strategy'])
imputador.set_params(strategy='median')
imputador.fit(datos)
print('Estrategia nueva:', imputador.get_params()['strategy'])
print('Valores aprendidos de nuevo:', imputador.statistics_)